# Notebook 2 — Data Preprocessing
Symptom encoding, cleaning, feature selection, missing value handling, and saving processed data.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '../'
OUT  = '../data/processed/'

train_dis  = pd.read_csv(f'{DATA}train_disease.csv')
test_dis   = pd.read_csv(f'{DATA}test_disease.csv')
drug_train = pd.read_csv(f'{DATA}UCIdrug_train.csv')
drug_test  = pd.read_csv(f'{DATA}UCIdrug_test.csv')

## 2.1 Disease Dataset Preprocessing

In [2]:
# --- Drop trailing empty column in train (exists as unnamed or empty) ---
train_dis.columns = train_dis.columns.str.strip()
test_dis.columns  = test_dis.columns.str.strip()

# Remove columns not in test (the trailing unnamed column)
common_cols = [c for c in train_dis.columns if c in test_dis.columns]
train_dis = train_dis[common_cols]
print('Aligned columns — train:', train_dis.shape, '  test:', test_dis.shape)

Aligned columns — train: (4920, 133)   test: (42, 133)


In [3]:
# --- Symptom columns are already binary (0/1) — verify ---
symptom_cols = [c for c in train_dis.columns if c != 'prognosis']
print('Unique values across symptom cols:', train_dis[symptom_cols].stack().unique())

Unique values across symptom cols: [1 0]


In [4]:
# --- Standardise disease label names (strip extra whitespace) ---
train_dis['prognosis'] = train_dis['prognosis'].str.strip()
test_dis['prognosis']  = test_dis['prognosis'].str.strip()
print('Disease classes after strip:', sorted(train_dis['prognosis'].unique()))

Disease classes after strip: ['(vertigo) Paroymsal  Positional Vertigo', 'AIDS', 'Acne', 'Alcoholic hepatitis', 'Allergy', 'Arthritis', 'Bronchial Asthma', 'Cervical spondylosis', 'Chicken pox', 'Chronic cholestasis', 'Common Cold', 'Dengue', 'Diabetes', 'Dimorphic hemmorhoids(piles)', 'Drug Reaction', 'Fungal infection', 'GERD', 'Gastroenteritis', 'Heart attack', 'Hepatitis B', 'Hepatitis C', 'Hepatitis D', 'Hepatitis E', 'Hypertension', 'Hyperthyroidism', 'Hypoglycemia', 'Hypothyroidism', 'Impetigo', 'Jaundice', 'Malaria', 'Migraine', 'Osteoarthristis', 'Paralysis (brain hemorrhage)', 'Peptic ulcer diseae', 'Pneumonia', 'Psoriasis', 'Tuberculosis', 'Typhoid', 'Urinary tract infection', 'Varicose veins', 'hepatitis A']


In [5]:
# --- Remove duplicate rows ---
before = len(train_dis)
train_dis.drop_duplicates(inplace=True)
print(f'Removed {before - len(train_dis)} duplicate rows from train_disease')

Removed 4616 duplicate rows from train_disease


In [6]:
# --- Feature selection: drop symptom columns with zero variance ---
zero_var = [c for c in symptom_cols if train_dis[c].std() == 0]
print('Zero-variance symptom cols to drop:', zero_var)
train_dis.drop(columns=zero_var, inplace=True)
test_dis.drop(columns=[c for c in zero_var if c in test_dis.columns], inplace=True)
symptom_cols = [c for c in train_dis.columns if c != 'prognosis']
print('Final symptom count:', len(symptom_cols))

Zero-variance symptom cols to drop: ['fluid_overload']
Final symptom count: 131


In [7]:
# --- Encode target label ---
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_dis['prognosis_enc'] = le.fit_transform(train_dis['prognosis'])
test_dis['prognosis_enc']  = le.transform(test_dis['prognosis'])

# Save label mapping
label_map = pd.DataFrame({'label': le.classes_, 'encoded': range(len(le.classes_))})
label_map.to_csv(f'{OUT}label_map.csv', index=False)
print(label_map)

                                      label  encoded
0   (vertigo) Paroymsal  Positional Vertigo        0
1                                      AIDS        1
2                                      Acne        2
3                       Alcoholic hepatitis        3
4                                   Allergy        4
5                                 Arthritis        5
6                          Bronchial Asthma        6
7                      Cervical spondylosis        7
8                               Chicken pox        8
9                       Chronic cholestasis        9
10                              Common Cold       10
11                                   Dengue       11
12                                 Diabetes       12
13             Dimorphic hemmorhoids(piles)       13
14                            Drug Reaction       14
15                         Fungal infection       15
16                                     GERD       16
17                          Gastroenteritis   

## 2.2 Drug Dataset Preprocessing

In [8]:
def clean_drug_df(df):
    # Drop corrupt condition rows (HTML artifacts)
    df = df[~df['condition'].str.contains('users found', na=True)].copy()
    df.dropna(subset=['condition', 'drugName', 'rating'], inplace=True)
    df.drop_duplicates(inplace=True)
    df['condition'] = df['condition'].str.strip()
    df['drugName']  = df['drugName'].str.strip()
    df['rating']    = pd.to_numeric(df['rating'], errors='coerce')
    df.dropna(subset=['rating'], inplace=True)
    # Parse date
    df['date'] = pd.to_datetime(df['date'], format='%d-%b-%y', errors='coerce')
    return df

drug_train = clean_drug_df(drug_train)
drug_test  = clean_drug_df(drug_test)
print('drug_train after cleaning:', drug_train.shape)
print('drug_test  after cleaning:', drug_test.shape)

drug_train after cleaning: (159498, 7)
drug_test  after cleaning: (53200, 7)


## 2.3 Disease-to-Condition Mapping
The drug dataset uses different condition names. We define a canonical mapping from the
41 disease prediction labels to the closest matching conditions in the drug dataset.
Diseases with no matching drug data are mapped to `None` and handled by the recommendation
model with a fallback rule.

In [9]:
DISEASE_TO_CONDITION = {
    '(vertigo) Paroymsal  Positional Vertigo': 'Vertigo',
    'AIDS':                          'HIV Infection',
    'Acne':                          'Acne',
    'Alcoholic hepatitis':           'Alcoholic Liver Disease',
    'Allergy':                       'Allergies',
    'Arthritis':                     'Rheumatoid Arthritis',
    'Bronchial Asthma':              'Asthma',
    'Cervical spondylosis':          'Cervical Spondylosis',
    'Chicken pox':                   'Chickenpox',
    'Chronic cholestasis':           'Primary Biliary Cholangitis',
    'Common Cold':                   'Cold Symptoms',
    'Dengue':                        'Dengue Fever',
    'Diabetes':                      'Diabetes, Type 2',
    'Dimorphic hemmorhoids(piles)':  'Hemorrhoids',
    'Drug Reaction':                 'Drug Hypersensitivity',
    'Fungal infection':              'Tinea Corporis',
    'GERD':                          'GERD',
    'Gastroenteritis':               'Gastroenteritis',
    'Heart attack':                  'Heart Attack',
    'Hepatitis B':                   'Hepatitis B',
    'Hepatitis C':                   'Hepatitis C',
    'Hepatitis D':                   'Hepatitis',
    'Hepatitis E':                   'Hepatitis',
    'Hypertension':                  'High Blood Pressure',
    'Hyperthyroidism':               'Hyperthyroidism',
    'Hypoglycemia':                  'Hypoglycemia',
    'Hypothyroidism':                'Hypothyroidism',
    'Impetigo':                      'Impetigo',
    'Jaundice':                      'Jaundice',
    'Malaria':                       'Malaria',
    'Migraine':                      'Migraine',
    'Osteoarthristis':               'Osteoarthritis',
    'Paralysis (brain hemorrhage)':  'Stroke',
    'Peptic ulcer diseae':           'Peptic Ulcer Disease',
    'Pneumonia':                     'Pneumonia',
    'Psoriasis':                     'Psoriasis',
    'Tuberculosis':                  'Tuberculosis, Active',
    'Typhoid':                       'Typhoid Fever',
    'Urinary tract infection':       'Urinary Tract Infection',
    'Varicose veins':                'Varicose Veins',
    'hepatitis A':                   'Hepatitis A',
}

mapping_df = pd.DataFrame(list(DISEASE_TO_CONDITION.items()),
                          columns=['disease', 'drug_condition'])
mapping_df.to_csv(f'{OUT}disease_condition_map.csv', index=False)
print(mapping_df)

                                    disease               drug_condition
0   (vertigo) Paroymsal  Positional Vertigo                      Vertigo
1                                      AIDS                HIV Infection
2                                      Acne                         Acne
3                       Alcoholic hepatitis      Alcoholic Liver Disease
4                                   Allergy                    Allergies
5                                 Arthritis         Rheumatoid Arthritis
6                          Bronchial Asthma                       Asthma
7                      Cervical spondylosis         Cervical Spondylosis
8                               Chicken pox                   Chickenpox
9                       Chronic cholestasis  Primary Biliary Cholangitis
10                              Common Cold                Cold Symptoms
11                                   Dengue                 Dengue Fever
12                                 Diabetes        

## 2.4 Save Processed Data

In [10]:
train_dis.to_csv(f'{OUT}train_disease_clean.csv', index=False)
test_dis.to_csv(f'{OUT}test_disease_clean.csv',   index=False)
drug_train.to_csv(f'{OUT}drug_train_clean.csv',   index=False)
drug_test.to_csv(f'{OUT}drug_test_clean.csv',     index=False)
print('All processed files saved to data/processed/')

All processed files saved to data/processed/
